<a href="https://colab.research.google.com/github/1kaiser/freecad-skill/blob/main/examples/FreeCAD_Ollama_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreeCAD + Ollama (Local LLM) in Google Colab 🚀

This notebook demonstrates how to run an autonomous 3D CAD modeling workflow using **FreeCAD** and a local **Ollama** LLM (e.g., Llama 3 or Mistral) completely within Google Colab.

## 1. Environment Setup
First, we'll install FreeCAD, X virtual framebuffer (`xvfb`) for headless execution, and python 3D rendering packages like `trimesh`.

In [1]:
!sudo apt update
!sudo add-apt-repository ppa:freecad-maintainers/freecad-stable -y
!sudo apt update
!sudo DEBIAN_FRONTEND=noninteractive apt install -y freecad xvfb npm
!npm install -g obj2gltf
!pip install trimesh pyrender numpy scipy pillow xvfbwrapper pyglet matplotlib

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,821 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

## 2. Setting up the Local LLM (Ollama)
We download and install Ollama, then run it in the background to serve as our agentic reasoning engine.

## 3. Interpreting User Instructions
We provide the LLM with the context of FreeCAD's Python API, followed by the user's instructions. The LLM translates the user's intent into executable Python code.

In [6]:
import subprocess
import time
import os
import requests
import json
import re

# --- STEP 1: SETUP & START SERVER ---
print("--- Initializing Ollama ---")
# 1. Install zstd and Ollama (only if not already installed)
!apt-get update -qq && apt-get install -y zstd -qq
if not os.path.exists("/usr/local/bin/ollama"):
    !curl -fsSL https://ollama.com/install.sh | sh

# 2. Kill any old instances to avoid port conflicts
!pkill ollama

# 3. Start server in the background
ollama_process = subprocess.Popen(['ollama', 'serve'],
                                  stdout=subprocess.DEVNULL,
                                  stderr=subprocess.DEVNULL)

# 4. Wait for server to wake up
time.sleep(5)

# 5. Pull the model (Using qwen2.5:7b as qwen3.5 doesn't exist yet)
MODEL_NAME = "qwen3.5:9b"
print(f"--- Pulling Model {MODEL_NAME} (This may take a few mins) ---")
subprocess.run(['ollama', 'pull', MODEL_NAME], check=True)


# --- STEP 2: DEFINE GENERATION LOGIC ---
def generate_freecad_script(user_instruction):
    system_prompt = """
    You are a FreeCAD Python API expert.
    Write a Python script that uses the `FreeCAD` and `Part` modules to generate the requested 3D model.
    Ensure you append `/usr/lib/freecad/lib` and `/usr/share/freecad/Mod` to `sys.path`.
    Export the result to 'model.stl' and 'model.obj'.
    Return ONLY valid Python code, no markdown wrappers, no explanations.
    """

    prompt = f"{system_prompt}\n\nUser Instruction: {user_instruction}\n\nPython Code:"

    # Call the local Ollama API
    try:
        response = requests.post("http://localhost:11434/api/generate",
                                 json={"model": MODEL_NAME, "prompt": prompt, "stream": False},
                                 timeout=120)
        if response.status_code == 200:
            return response.json()["response"]
        else:
            return f"Error: {response.status_code}"
    except Exception as e:
        return f"Connection Error: {e}"


# --- STEP 3: EXECUTION ---
user_instruction = "Create a 3D tree. The trunk should be a cylinder with radius 5 and height 50. Make sure the tree is standing upright on the ground along the Z-axis. The foliage should consist of three spheres on top of the trunk."

print("--- Generating Python Script ---")
generated_code = generate_freecad_script(user_instruction)

# Clean the output (remove markdown if the LLM ignored instructions)
cleaned_code = re.sub(r'```python|```', '', generated_code).strip()

print("--- Result ---")
print(cleaned_code)

# Save to file
with open("generated_freecad_script.py", "w") as f:
    f.write(cleaned_code)

print("\n--- DONE: Script saved to 'generated_freecad_script.py' ---")

# --- STEP 4: CLEANUP (SHUTDOWN) ---
print("--- Shutting down Ollama server ---")
ollama_process.terminate()
!pkill ollama

--- Initializing Ollama ---
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--- Pulling Model qwen3.5:9b (This may take a few mins) ---
--- Generating Python Script ---
--- Result ---
Connection Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)

--- DONE: Script saved to 'generated_freecad_script.py' ---
--- Shutting down Ollama server ---


In [14]:
import subprocess
import time
import os
import requests
import re

# --- CONFIGURATION ---
MODEL_NAME = "qwen2.5:1.5b"

# --- STEP 1: START SERVER ---
print("--- 1. Starting Ollama Server ---")
!pkill ollama
time.sleep(2)
ollama_process = subprocess.Popen(['ollama', 'serve'],
                                  stdout=subprocess.DEVNULL,
                                  stderr=subprocess.DEVNULL)

# Wait for server to wake up
for i in range(15):
    try:
        if requests.get("http://localhost:11434/", timeout=2).status_code == 200:
            print("Server is UP!")
            break
    except:
        time.sleep(2)

# --- STEP 2: GENERATE CODE WITH HELPER FUNCTIONS ---
def generate_freecad_script(user_instruction):
    # We provide the AI with simple helper functions so it doesn't mess up the Vector syntax
    system_prompt = """
    You are a FreeCAD Python expert.
    Use ONLY these helper functions to build the model:

    def create_cylinder(radius, height, z_pos):
        c = Part.makeCylinder(radius, height)
        c.Placement.Base = FreeCAD.Vector(0, 0, z_pos)
        return c

    def create_sphere(radius, x, y, z):
        s = Part.makeSphere(radius)
        s.Placement.Base = FreeCAD.Vector(x, y, z)
        return s

    Rules:
    1. Start with 'import FreeCAD, Part, Mesh'.
    2. Add the two helper functions above to the script.
    3. Use the helpers to create the trunk and foliage.
    4. At the end, use Mesh.export([list_of_objects], "model.stl").
    5. Return ONLY Python code.
    """

    prompt = f"{system_prompt}\n\nUser Instruction: {user_instruction}\n\nPython Code:"

    print("--- 2. Generating Script ---")
    try:
        response = requests.post("http://localhost:11434/api/generate",
                                 json={"model": MODEL_NAME, "prompt": prompt, "stream": False},
                                 timeout=300)
        return response.json().get("response", "")
    except Exception as e:
        return f"API Error: {e}"

user_instruction = "Create a tree. Trunk: cylinder radius 5, height 50. Foliage: 3 spheres radius 15 at the top of the trunk."
generated_code = generate_freecad_script(user_instruction)

# Clean code and ensure sys.path is included
header = """import sys
sys.path.append('/usr/lib/freecad/lib')
sys.path.append('/usr/share/freecad/Mod')
import FreeCAD, Part, Mesh
"""
cleaned_body = re.sub(r'```python|```', '', generated_code).strip()
final_script = header + "\n" + cleaned_body

with open("generated_freecad_script.py", "w") as f:
    f.write(final_script)

print("\n--- GENERATED SCRIPT ---")
print(final_script)

# --- STEP 3: RUN FREECAD ---
print("\n--- 3. Executing FreeCAD ---")
# Ensure FreeCAD is installed
if subprocess.run(['which', 'freecadcmd'], capture_output=True).returncode != 0:
    print("Installing FreeCAD...")
    !apt-get update -qq && apt-get install -y freecad-common freecad-python3 > /dev/null

# Execute
!freecadcmd -c generated_freecad_script.py

# Final result check
if os.path.exists("model.stl"):
    print("\n✅ SUCCESS: 'model.stl' created!")
    from google.colab import files
    # files.download('model.stl') # Optional
else:
    print("\n❌ FAILED: The script output above shows the error.")

# Cleanup
ollama_process.terminate()
!pkill ollama

--- 1. Starting Ollama Server ---
Server is UP!
--- 2. Generating Script ---

--- GENERATED SCRIPT ---
import sys
sys.path.append('/usr/lib/freecad/lib')
sys.path.append('/usr/share/freecad/Mod')
import FreeCAD, Part, Mesh

import FreeCAD, Part, Mesh

# Create the trunk
trunk_radius = 5
trunk_height = 50
trunk_z_pos = 0

trunk = create_cylinder(trunk_radius, trunk_height, trunk_z_pos)

# Create the foliage
num_spheres = 3
sphere_radius = 15
sphere_x = 0
sphere_y = 0
sphere_z = 0

foliage = []
for i in range(num_spheres):
    foliage.append(create_sphere(sphere_radius, sphere_x, sphere_y, sphere_z))

# Export the model
Mesh.export(foliage, "tree.stl")

--- 3. Executing FreeCAD ---
Exception while processing file: generated_freecad_script.py [name 'create_cylinder' is not defined]
[FreeCAD Console mode <Use Ctrl-D (i.e. EOF) to exit.>]
>>> 
KeyboardInterrupt
>>> 
KeyboardInterrupt
>>> ^C

❌ FAILED: The script output above shows the error.


In [15]:
import subprocess
import time
import os
import requests
import re

# --- CONFIGURATION ---
MODEL_NAME = "qwen2.5:1.5b"
MAX_RETRIES = 3

# --- STEP 1: INITIALIZE SERVER ---
print("--- 1. Initializing Environment ---")
!pkill ollama
time.sleep(2)
ollama_process = subprocess.Popen(['ollama', 'serve'],
                                  stdout=subprocess.DEVNULL,
                                  stderr=subprocess.DEVNULL)

# Wait for server
for i in range(15):
    try:
        if requests.get("http://localhost:11434/", timeout=2).status_code == 200:
            print("Server is UP!")
            break
    except:
        time.sleep(2)

# --- STEP 2: DEFINE AGENT FUNCTIONS ---
def ask_ai(prompt):
    try:
        response = requests.post("http://localhost:11434/api/generate",
                                 json={"model": MODEL_NAME, "prompt": prompt, "stream": False},
                                 timeout=300)
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"API Error: {e}"

def clean_code(text):
    # Remove markdown formatting if present
    code = re.sub(r'```python|```', '', text).strip()
    # Ensure sys.path headers are present
    header = "import sys\nsys.path.append('/usr/lib/freecad/lib')\nsys.path.append('/usr/share/freecad/Mod')\nimport FreeCAD, Part, Mesh\n"
    if "import FreeCAD" not in code:
        return header + "\n" + code
    return code

# --- STEP 3: THE SELF-CORRECTION LOOP ---
user_instruction = "Create a 3D tree. Trunk: cylinder radius 5, height 50. Foliage: 3 spheres radius 15 at the top of the trunk. Export to model.stl"

current_prompt = f"""
You are a FreeCAD Python expert. Write a script to: {user_instruction}
Rules:
1. Use FreeCAD.Vector(x,y,z) for positions.
2. Use Mesh.export([list_of_objects], 'model.stl') for export.
3. Return ONLY valid Python code. No explanations.
"""

for attempt in range(MAX_RETRIES):
    print(f"\n--- Attempt {attempt + 1} ---")

    # 1. Get code from AI
    raw_ai_output = ask_ai(current_prompt)
    script_content = clean_code(raw_ai_output)

    with open("generated_freecad_script.py", "w") as f:
        f.write(script_content)

    # 2. Try to run it in FreeCAD
    print("Executing FreeCAD...")
    # Capture the output so we can read the error
    result = subprocess.run(['freecadcmd', '-c', 'generated_freecad_script.py'],
                            capture_output=True, text=True)

    # 3. Check for errors
    # Note: FreeCAD often outputs errors to stdout or contains "Exception"
    if "Exception" in result.stdout or "Error" in result.stdout or result.returncode != 0:
        error_msg = result.stdout if "Exception" in result.stdout else result.stderr
        print(f"❌ Execution Failed! Error detected:\n{error_msg[:200]}...")

        # 4. Prepare the prompt for the next attempt
        current_prompt = f"""
        Your previous code failed with this error: {error_msg}

        Previous Code:
        {script_content}

        Please rewrite the entire script to fix the error.
        Ensure you use correct FreeCAD API syntax (e.g. Part.makeCylinder(radius, height) takes only 2 or 3 arguments).
        Return ONLY the corrected Python code.
        """
    else:
        if os.path.exists("model.stl"):
            print("✅ SUCCESS: model.stl created!")
            break
        else:
            print("❓ Script finished but model.stl was not found. Retrying...")
            current_prompt += "\nNote: The script ran but did not save the 'model.stl' file. Check your export logic."

# --- STEP 4: CLEANUP ---
ollama_process.terminate()
!pkill ollama
print("\n--- Workflow Complete ---")

--- 1. Initializing Environment ---
Server is UP!

--- Attempt 1 ---
Executing FreeCAD...
❓ Script finished but model.stl was not found. Retrying...

--- Attempt 2 ---
Executing FreeCAD...
❓ Script finished but model.stl was not found. Retrying...

--- Attempt 3 ---
Executing FreeCAD...
❓ Script finished but model.stl was not found. Retrying...

--- Workflow Complete ---


## 4. Execute FreeCAD Headless
We run the generated script using `freecadcmd` to produce our 3D artifacts.

In [12]:
!freecadcmd -c generated_freecad_script.py

Exception while processing file: generated_freecad_script.py [argument 2 must be Base.Vector, not tuple]
[FreeCAD Console mode <Use Ctrl-D (i.e. EOF) to exit.>]
>>> 
KeyboardInterrupt
>>> 
KeyboardInterrupt
>>> ^C


## 5. Render Output inside Colab
Finally, we can load the exported `.stl` file using `trimesh` and render it right here in the notebook.

In [ ]:
import trimesh
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

mesh = trimesh.load("model.stl")
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

faces = mesh.faces
vertices = mesh.vertices

# Color the tree green
poly3d = Poly3DCollection(vertices[faces], facecolors='#228B22', alpha=0.9, linewidths=0)
ax.add_collection3d(poly3d)

# Scale the plot axes evenly
max_range = np.array([mesh.bounds[1][0]-mesh.bounds[0][0],
                      mesh.bounds[1][1]-mesh.bounds[0][1],
                      mesh.bounds[1][2]-mesh.bounds[0][2]]).max() / 2.0
mid_x = (mesh.bounds[1][0]+mesh.bounds[0][0]) * 0.5
mid_y = (mesh.bounds[1][1]+mesh.bounds[0][1]) * 0.5
mid_z = (mesh.bounds[1][2]+mesh.bounds[0][2]) * 0.5

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

ax.grid(False)
plt.axis('off')
plt.title("Generated FreeCAD Tree Model")
plt.show()